# Organoid sorter — simulation driver

Mirrors `image_analysis.ipynb`, but:

- `BurstSimulator` reproduces flow-chamber behavior: organoids enter from the
  left, **translate across the FOV** `step_px` pixels per frame, and exit on
  the right. Each is visible for ~2–3 frames. Events are spaced by `gap`
  empty frames. Most events have a single organoid; some have two.
- `ThresholdSegmentationModel` stands in for Convpaint (bright pixels → `class_id=2`).
- All downstream code (`clean_and_label`, `measure_regions`, `ParticleTracker`,
  sort decision, parquet logging, motor/pump stubs) is the real `rtm` code.

**New:** a **"multiple"** sort category. When two or more organoids are visible
simultaneously in the same event, they are routed to `MULTI_POSITION` instead
of being size-sorted into positions 1/2/3.


## 1. Config


In [ ]:
import shutil
from pathlib import Path

STORAGE_PATH = "output_sim"
IMG_SHAPE = (512, 512)   # (H, W)
BACKGROUND = 20          # uniform background intensity
DOT_VALUE = 200          # organoid intensity

# Start clean so parquet files don't accumulate across re-runs.
if Path(STORAGE_PATH).exists():
    shutil.rmtree(STORAGE_PATH)
Path(STORAGE_PATH).mkdir(parents=True, exist_ok=True)


## 2. Flow-chamber simulator

Each entry in `SCHEDULE` is one "event": one or more organoids that enter
from the left at the same moment, translate together at `step_px` pixels per
frame, and exit on the right. After every organoid in the event has left the
FOV, `gap` empty frames pass before the next event begins.

Number of frames a given organoid is visible ≈ `(W + 2r) / step_px`. With the
default `step_px = 180` and `W = 512`, that's ~3 frames for radii 8–28 px —
matching the "2–3 frame transit" case we're targeting. The tracker's
`SEARCH_RANGE` must be ≥ `step_px` to link consecutive frames, so bump it
together with `step_px` if you change the speed.


In [ ]:
import numpy as np


class BurstSimulator:
    """Flow-chamber simulator.

    Each entry in *schedule* triggers one event: one or more organoids enter
    from the left together, translate ``step_px`` pixels/frame, and exit on
    the right. Number of visible frames per organoid ≈ ``(W + 2r) / step_px``.

    schedule entry::

        {"radii": [r1, r2, ...], "gap": 3}

    ``gap`` is the number of empty frames after the event (before the next one).
    Multiple radii in one entry = multi-organoid event — they share `y`-bands
    and move lockstep.
    """

    def __init__(self, shape, background, dot_value, schedule, step_px=180):
        self.shape = shape
        self.bg = np.uint8(background)
        self.fg = np.uint8(dot_value)
        self.step_px = int(step_px)
        H, W = shape

        # Flatten schedule into per-frame instructions: list of (x, y, r) per frame.
        self._frames: list[list[tuple[int, int, int]]] = []
        for entry in schedule:
            radii = list(entry["radii"])
            gap = int(entry.get("gap", 3))
            n = len(radii)
            max_r = max(radii)
            # Center starts just off-screen left so the first visible frame
            # already has a substantial chunk of the disk inside the FOV.
            x0 = -max_r
            t_local = 0
            while True:
                dots = []
                any_visible = False
                for i, r in enumerate(radii):
                    y = int(H * (i + 1) / (n + 1))   # evenly spaced y-bands
                    x = x0 + self.step_px * t_local
                    # Visible when the disk intersects the FOV.
                    if x - r <= W - 1 and x + r >= 0:
                        dots.append((int(x), int(y), int(r)))
                        any_visible = True
                if not any_visible:
                    break
                self._frames.append(dots)
                t_local += 1
            for _ in range(gap):
                self._frames.append([])
        self._idx = 0

    def snap(self) -> np.ndarray:
        img = np.full(self.shape, self.bg, dtype=np.uint8)
        if self._idx >= len(self._frames):
            return img
        dots = self._frames[self._idx]
        self._idx += 1
        H, W = self.shape
        yy, xx = np.ogrid[:H, :W]
        for x, y, r in dots:
            mask = (yy - y) ** 2 + (xx - x) ** 2 <= r * r
            img[mask] = self.fg
        return img

    @property
    def done(self) -> bool:
        return self._idx >= len(self._frames)


SCHEDULE = [
    {"radii": [16],     "gap": 3},   # 1 medium
    {"radii": [8],      "gap": 3},   # 1 small
    {"radii": [28],     "gap": 3},   # 1 large
    {"radii": [16, 8],  "gap": 3},   # MULTI
    {"radii": [28],     "gap": 3},   # 1 large
    {"radii": [8],      "gap": 3},   # 1 small
    {"radii": [16, 28], "gap": 3},   # MULTI
    {"radii": [16],     "gap": 3},   # 1 medium
]


def make_sim():
    return BurstSimulator(
        shape=IMG_SHAPE,
        background=BACKGROUND,
        dot_value=DOT_VALUE,
        schedule=SCHEDULE,
        step_px=180,
    )


## 3. Fake segmentation model

Drop-in replacement for `rtm.segmentation.SegmentationModel`. The run loop
only calls `.segment(image) -> class_labels` on its model, so a simple
threshold suffices to feed the real `clean_and_label` + `measure_regions`
path.


In [ ]:
class ThresholdSegmentationModel:
    def __init__(self, threshold=100, class_id=2):
        self.threshold = threshold
        self.class_id = class_id

    def segment(self, image: np.ndarray) -> np.ndarray:
        out = np.zeros(image.shape, dtype=np.int32)
        out[image > self.threshold] = self.class_id
        return out


model = ThresholdSegmentationModel(threshold=BACKGROUND + 50, class_id=2)


## 4. Preview one frame

Advance the simulator a few frames and show raw + segmentation so you can
confirm the burst shape before kicking off the main loop.


In [ ]:
import matplotlib.pyplot as plt

# Advance a couple of snaps so the first organoid is mid-FOV.
preview = make_sim()
for _ in range(2):
    img = preview.snap()
labels = model.segment(img)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(img, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Simulated image (mid-flow)")
axes[1].imshow(labels, cmap="nipy_spectral")
axes[1].set_title(f"Class labels (class-2 pixels = {int((labels == 2).sum())})")
plt.tight_layout()


## 5. Run the sorter — snap → segment → clean → track → sort

Same structure as the main-notebook run cell, but:

- input frames come from `make_sim()` instead of the microscope;
- a **multi-organoid detector** flags any particle ever seen *simultaneously*
  with another particle during its track; flagged tracks are routed to
  `MULTI_POSITION` (eppendorf slot 4) regardless of their size.
- loop exits when the simulator is exhausted; a final `flush` emits any
  tracks still active at that moment.


In [ ]:
import time
import os
import tifffile
import pandas as pd
from IPython.display import clear_output

from rtm.segmentation import clean_and_label, measure_regions
from rtm.tracking import ParticleTracker
from rtm.motor import actuate_pump, move_eppendorf, position_from_size
from rtm.persistence import DetectionLog, save_detection_image

# ---- Knobs --------------------------------------------------------------
tryout = "sim01"
PREVIEW_INTERVAL_S = 0.3

CLASS_ID = 2
MIN_PIXEL_SIZE = 10
HOLE_AREA_THRESHOLD = 10

# Tracking: SEARCH_RANGE must be >= simulator step_px so fast flow still links.
SEARCH_RANGE = 220.0
MEMORY = 1
WINDOW_FRAMES = 30
MIN_TRACK_LENGTH = 2

DETECTION_AREA_THRESHOLD = 50.0
SIZE_TO_POSITION = [
    (400.0, 1),         # small
    (1500.0, 2),        # medium
    (float("inf"), 3),   # large
]
MULTI_POSITION = 4       # eppendorf slot for events with >1 organoid

# ---- State --------------------------------------------------------------
tracker = ParticleTracker(search_range=SEARCH_RANGE, memory=MEMORY, window_frames=WINDOW_FRAMES)
sorts_log = DetectionLog(f"{STORAGE_PATH}/sorts.parquet")
obs_log = DetectionLog(f"{STORAGE_PATH}/observations.parquet")
best_images: dict = {}    # particle_id -> (best_area, best_frame, best_image)
multi_flag: dict = {}     # particle_id -> True if ever seen simultaneously with another particle

os.makedirs(f"{STORAGE_PATH}/rawstream", exist_ok=True)
os.makedirs(f"{STORAGE_PATH}/detections", exist_ok=True)


def emit_sort(grp, best, is_multi):
    """Build one row for sorts.parquet (or None if the track is filtered out)."""
    if len(grp) < MIN_TRACK_LENGTH:
        return None
    mean_area = float(grp["area"].mean())
    if mean_area <= DETECTION_AREA_THRESHOLD and not is_multi:
        return None
    pid = int(grp["particle"].iloc[0])
    pos = MULTI_POSITION if is_multi else position_from_size(mean_area, SIZE_TO_POSITION)
    best_frame = int(best[1]) if best else int(grp["frame"].iloc[-1])
    image_file = save_detection_image(
        best[2], f"{STORAGE_PATH}/detections",
        timestep=best_frame, suffix=f"_p{pid:05d}",
    ) if best else ""
    move_eppendorf(pos)
    return {
        "particle_id": pid,
        "t_first": int(grp["frame"].min()),
        "t_last": int(grp["frame"].max()),
        "n_frames": int(len(grp)),
        "mean_area": mean_area,
        "max_area": float(grp["area"].max()),
        "mean_x": float(grp["x"].mean()),
        "mean_y": float(grp["y"].mean()),
        "is_multi": bool(is_multi),
        "sort_position": int(pos),
        "image_file": image_file,
        "timestamp": time.time(),
    }


sim = make_sim()
t = 0
try:
    while not sim.done:
        img = sim.snap()

        # --- Segment + clean + label + measure ---------------------------
        class_labels = model.segment(img)
        cc_labels = clean_and_label(
            class_labels, class_id=CLASS_ID,
            min_pixel_size=MIN_PIXEL_SIZE, hole_area_threshold=HOLE_AREA_THRESHOLD,
        )
        regions = measure_regions(cc_labels)

        # --- Track -------------------------------------------------------
        tracks = tracker.update(regions, timestep=t)

        if not tracks.empty:
            this_frame = tracks[tracks["frame"] == t]
            # Multi detection: if ≥2 distinct particles this frame, flag them all forever.
            pids_now = this_frame["particle"].unique().tolist()
            if len(pids_now) >= 2:
                for pid in pids_now:
                    multi_flag[int(pid)] = True
            for _, row in this_frame.iterrows():
                pid, area = int(row["particle"]), float(row["area"])
                entry = best_images.get(pid)
                if entry is None or area > entry[0]:
                    best_images[pid] = (area, t, img.copy())
            obs_log.add(this_frame.assign(timestamp=time.time()))

        # --- Pump ---------------------------------------------------------
        speed = tracker.flow_speed(tracks)
        if speed is not None:
            actuate_pump(int(round(speed)))

        # --- Sort completed tracks ----------------------------------------
        completed = tracker.pop_completed_tracks(tracks, timestep=t)
        sort_rows = []
        for pid, grp in completed.groupby("particle", sort=False):
            pid = int(pid)
            row = emit_sort(grp, best_images.pop(pid, None), multi_flag.pop(pid, False))
            if row is not None:
                sort_rows.append(row)
        if sort_rows:
            sorts_log.add(pd.DataFrame(sort_rows))

        # --- Display ------------------------------------------------------
        areas = regions["area"].astype(int).tolist() if not regions.empty else []
        areas_str = ", ".join(str(a) for a in areas) or "none"
        speed_str = f"{speed:.2f}" if speed is not None else "--"

        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(18, 9))
        axes[0].imshow(img, cmap="gray", vmin=0, vmax=255)
        axes[0].set_title(f"Raw (sim)  (t={t}, flow={speed_str} px/f, sorted+={len(sort_rows)})", fontsize=14)
        axes[1].imshow(cc_labels, cmap="nipy_spectral")
        axes[1].set_title(f"Labels ({len(areas)} object(s)) — areas: {areas_str}", fontsize=14)
        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        plt.show()
        if not regions.empty:
            print(regions.to_string(index=False))

        tifffile.imwrite(f"{STORAGE_PATH}/rawstream/{tryout}_{t:03d}.tiff", img)
        t += 1
        time.sleep(PREVIEW_INTERVAL_S)

    # --- Flush any still-active tracks at simulator end ------------------
    far_future = t + WINDOW_FRAMES + 1
    flush_tracks = tracker.update(pd.DataFrame(), timestep=far_future)
    completed = tracker.pop_completed_tracks(flush_tracks, timestep=far_future)
    flushed_rows = []
    for pid, grp in completed.groupby("particle", sort=False):
        pid = int(pid)
        row = emit_sort(grp, best_images.pop(pid, None), multi_flag.pop(pid, False))
        if row is not None:
            flushed_rows.append(row)
    if flushed_rows:
        sorts_log.add(pd.DataFrame(flushed_rows))
    print(f"simulator exhausted. flushed {len(flushed_rows)} late sort(s).")
except KeyboardInterrupt:
    print("stopped")


## 6. Inspect the sort log


In [ ]:
import pandas as pd

sorts = pd.read_parquet(f"{STORAGE_PATH}/sorts.parquet")
print(f"{len(sorts)} organoid(s) sorted")
if not sorts.empty:
    print("by sort_position:")
    print(sorts["sort_position"].value_counts().sort_index().to_string())
    print(f"\nmulti rows: {int(sorts['is_multi'].sum())}")
sorts


In [ ]:
obs_path = Path(f"{STORAGE_PATH}/observations.parquet")
if obs_path.exists():
    obs = pd.read_parquet(obs_path)
    print(f"{len(obs)} observation rows across {obs['frame'].nunique()} frames")
    display(obs.head(20))
